# Model Training
Train a lightweight CNN on single-target classes and save the model.

In [1]:
from pathlib import Path
import sys
import torch
from torch.utils.data import DataLoader, random_split

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.config import METADATA_CSV
from src.data.metadata import load_metadata, filter_by_set, material_label_map
from src.data.dataset import TWITorchDataset
from src.models.cnn1d import CNN1D
from src.models.trainer import Trainer, TrainConfig
from src.processing.preprocess import preprocess_signal

base_folder = ROOT / "data" / "36"
mat_key = "dataMeasured1"

metadata = load_metadata(METADATA_CSV)
metadata = filter_by_set(metadata, "single")
metadata = metadata[metadata["material"].notna()]
label_map = material_label_map(metadata)

file_paths = []
labels = []
for _, row in metadata.iterrows():
    mat_path = base_folder / str(int(row["folder_id"])) / "data.mat"
    if mat_path.exists():
        file_paths.append(mat_path)
        labels.append(label_map[row["material"]])

dataset = TWITorchDataset(file_paths, labels, mat_key=mat_key, transform=preprocess_signal)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_set, _ = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_set, batch_size=8, shuffle=True)

sample_x, _ = dataset[0]
model = CNN1D(in_channels=int(sample_x.shape[0]), num_classes=len(label_map))
trainer = Trainer(model, torch.device("cpu"), TrainConfig(epochs=2, learning_rate=1e-3))
metrics = trainer.fit(train_loader)
print("Train loss:", metrics)

out = ROOT / "outputs" / "models"
out.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), out / "best_model_nb.pt")
print("Saved model to outputs/models/best_model_nb.pt")

Train loss: {'loss': 1.1240590810775757}
Saved model to outputs/models/best_model_nb.pt
